# Federated Learning for Brain Tumor Classification with PIDL

This notebook runs the **full pipeline** with **real cryptographic secure aggregation** (Flower SecAgg+).

- **ResNet-18** backbone + **PIDL loss** (Perona-Malik regularization)
- **True encryption**: Flower SecAgg+ (secret-sharing, no simulation)
- Stratified data split, centralized evaluation on a held-out test set
- Logs saved to CSV for plotting

**Flow**: Mount Drive → Clone repo → Install → Run `flwr run .` (SecAgg+) → Plot results.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository (use your GitHub URL) and go into project root
# If you already have the repo under /content, skip clone and set PROJECT_DIR accordingly.
import os
REPO_URL = "https://github.com/yourusername/brain-tumor-classification-PIDL-FL.git"  # ← set your repo URL
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 {REPO_URL} /content/brain-tumor-classification-PIDL-FL
%cd {PROJECT_DIR}

In [ ]:
# Install project and dependencies (includes flwr[simulation]>=1.25.0 for SecAgg+)
# Run from project root so "pip install -e ." finds pyproject.toml
!cd {PROJECT_DIR} && pip install -e .

## Configuration

In [ ]:
# Configuration — used for flwr run and for plotting
DATA_ROOT = "/content/drive/MyDrive/PhysNet/datasets/brain_tumor_mri"
LOG_DIR   = "/content/drive/MyDrive/PhysNet/results/fl_brain_mri"
NUM_ROUNDS = 10
LOCAL_EPOCHS = 5
MIN_CLIENTS = 3

## Run Federated Learning

In [ ]:
# Run federated learning with real cryptographic secure aggregation (Flower SecAgg+)
# Full pipeline: FL rounds, SecAgg+ masking/unmasking, server logs to LOG_DIR (fl_rounds.csv, fl_clients.csv).
import os
os.makedirs(LOG_DIR, exist_ok=True)
_run_config = (
    f'"data-root={DATA_ROOT}" '
    f'"log-dir={LOG_DIR}" '
    f'"num-server-rounds={NUM_ROUNDS}" '
    f'"local-epochs={LOCAL_EPOCHS}" '
    f'"min-fit-clients={MIN_CLIENTS}" '
    '"is-demo=false"'
)
# Run from project root (PROJECT_DIR from cell 2)
!cd {PROJECT_DIR} && flwr run . --run-config {_run_config}

## Plot Results

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

log_dir = LOG_DIR
rounds_path = os.path.join(log_dir, "fl_rounds.csv")
clients_path = os.path.join(log_dir, "fl_clients.csv")

if not os.path.isfile(rounds_path) or not os.path.isfile(clients_path):
    print("No results yet. Run the previous cell (flwr run) first. If you already ran it, check that LOG_DIR exists and that the SecAgg+ app wrote CSVs there.")
else:
    rounds_df = pd.read_csv(rounds_path)
    clients_df = pd.read_csv(clients_path)

    # Global test accuracy over rounds
    plt.figure(figsize=(10, 6))
    plt.plot(rounds_df["round"], rounds_df["global_test_acc"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Accuracy (%)")
    plt.title("Global Test Accuracy over FL Rounds (SecAgg+)")
    plt.grid(True)
    plt.show()

    # Client training accuracies
    plt.figure(figsize=(12, 6))
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_acc"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Accuracy (%)")
    plt.title("Client Training Accuracies over FL Rounds")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Losses
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(rounds_df["round"], rounds_df["global_test_loss"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Loss")
    plt.title("Global Test Loss")
    plt.grid(True)
    plt.subplot(1, 2, 2)
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_loss"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Loss")
    plt.title("Client Training Losses")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()